# SafeHire Trust & Risk Assessment AI – Week 5 Exercise (winniekariuki)

**SafeHire AI Risk Advisor** — A decision-support tool for parents. Evaluates hiring risk of domestic workers using verified knowledge and nanny profiles. Helps answer: **"Is this nanny safe to hire?"**

In [ ]:
import json
import os
from dotenv import load_dotenv

import gradio as gr

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.messages import HumanMessage

load_dotenv()

In [ ]:
# Load Knowledge Documents
KB_PATH = os.path.normpath(os.path.join(os.getcwd(), "knowledge-base"))
if not os.path.isdir(KB_PATH):
    KB_PATH = os.path.normpath(os.path.join(os.path.dirname(os.getcwd()), "knowledge-base"))

loader = DirectoryLoader(KB_PATH, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)

print(f"Loaded {len(documents)} documents, split into {len(chunks)} chunks.")

In [ ]:
# Create Embeddings + DB
embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Vector store ready.")

In [ ]:
# Load Nanny Profiles
PROFILES_PATH = os.path.normpath(os.path.join(os.getcwd(), "profiles", "nanny_profiles.json"))
if not os.path.isfile(PROFILES_PATH):
    PROFILES_PATH = os.path.normpath(os.path.join(os.path.dirname(os.getcwd()), "profiles", "nanny_profiles.json"))

with open(PROFILES_PATH) as f:
    profiles = json.load(f)

def find_profile(query):
    for profile in profiles:
        if profile["name"].lower() in query.lower():
            return profile
    return None

print(f"Loaded {len(profiles)} nanny profiles.")

In [ ]:
# LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# RAG Logic
def ask_safehire(question):
    docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in docs])

    profile = find_profile(question)
    profile_text = ""
    if profile:
        profile_text = f"""
NANNY PROFILE:

Name: {profile['name']}
ID Verified: {profile['id_verified']}
References: {profile['references']}
Employment History: {profile['employment_history_months']} months
Previous Jobs: {profile['previous_jobs']}
Notes: {profile['notes']}
"""

    prompt = f"""
You are SafeHire AI, an assistant that helps families hire domestic workers safely.

Use ONLY the SafeHire knowledge base below.

SAFEHIRE KNOWLEDGE:
{context}

{profile_text}

Question: {question}

Provide a helpful and clear answer.
"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content

In [ ]:
# Gradio UI
def chat(question):
    return ask_safehire(question)

# Wider output for readability; output stacked below input
css = """
.gr-box { min-width: 640px !important; max-width: 900px !important; }
.gradio-container { max-width: 960px !important; }
"""

with gr.Blocks(title="SafeHire AI Risk Advisor", css=css) as demo:
    gr.Markdown("## SafeHire AI Risk Advisor")
    gr.Markdown("AI assistant helping families hire domestic workers safely. Ask about a nanny by name (e.g. Jane Wanjiku, Mary Atieno, Grace Njeri) for a profile-based risk assessment.")
    inp = gr.Textbox(label="Your question", placeholder="Ask about hiring best practices, red flags, or a specific nanny...")
    out = gr.Textbox(label="Response", lines=10)
    inp.submit(chat, inputs=inp, outputs=out)

demo.launch(inbrowser=True)